# Fakeddit Image Downloader (Colab)

This notebook downloads Reddit images for the Fakeddit dataset and saves them directly to your Google Drive.

**Before running:**
1. Make sure your Fakeddit TSV files are already in your Google Drive at:
   `My Drive/fakeddit/data/`
   - `multimodal_train.tsv`
   - `multimodal_validate.tsv`
   - `multimodal_test_public.tsv`
2. Run cells top to bottom.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
image_dir = '/content/drive/MyDrive/fakeddit/images'
print(len(os.listdir(image_dir)))

15048


## 2. Confirm TSV files are present

In [3]:
import os

DATA_DIR = '/content/drive/MyDrive/fakeddit/data'
IMAGE_DIR = '/content/drive/MyDrive/fakeddit/images'

os.makedirs(IMAGE_DIR, exist_ok=True)

print('Files in data dir:')
for f in os.listdir(DATA_DIR):
    print(' ', f)

Files in data dir:
  multimodal_test_public.tsv
  multimodal_train.tsv
  multimodal_validate.tsv


## 3. Install dependencies

In [4]:
!pip install -q tqdm pandas numpy

In [5]:
!pip install torch transformers scikit-learn tqdm pandas

In [6]:
!git clone https://github.com/Girish4679/FakeNewsDetector.git /content/FakeNewsDetector
%cd /content/FakeNewsDetector

Cloning into '/content/FakeNewsDetector'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (37/37), done.
Receiving objects: 100% (49/49), 56.00 KiB | 1005.00 KiB/s, done.
remote: Total 49 (delta 12), reused 41 (delta 7), pack-reused 0 (from 0)
Resolving deltas: 100% (12/12), done.
/content/FakeNewsDetector


In [8]:
!python src/model.py

Loading CLIP (this downloads weights on first run ~600MB)...
Loading weights: 100% 398/398 [00:00<00:00, 898.85it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Traceback (most recent call last):
  File "/content/FakeNewsDetector/src/model.py", line 223, in <module>
    logits = model(input_ids, attention_mask, pixel_values)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/us

In [9]:
import re

path = "/content/FakeNewsDetector/src/model.py"
with open(path, "r") as f:
    content = f.read()

old_encode_text = """    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        features = self.clip.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        return F.normalize(features, dim=-1)   # (B, 512)

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        features = self.clip.get_image_features(pixel_values=pixel_values)
        return F.normalize(features, dim=-1)   # (B, 512)"""

new_encode_text = """    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        features = self.clip.text_projection(out.pooler_output)
        return F.normalize(features, dim=-1)   # (B, 512)

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        out = self.clip.vision_model(pixel_values=pixel_values)
        features = self.clip.visual_projection(out.pooler_output)
        return F.normalize(features, dim=-1)   # (B, 512)"""

content = content.replace(old_encode_text, new_encode_text)
with open(path, "w") as f:
    f.write(content)

print("Done — model.py patched")

Done — model.py patched


In [10]:
!python src/model.py

Loading CLIP (this downloads weights on first run ~600MB)...
Loading weights: 100% 398/398 [00:00<00:00, 829.29it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Logits: tensor([0.1304, 0.1283, 0.1286, 0.1276])
Probs:  tensor([0.5326, 0.5320, 0.5321, 0.5319])

Total params:     149,899,906
Trainable params: 279,169  (0.2%)

Expected: ~150M total, ~200K trainable (frozen CLIP, MLP only)


In [16]:
!mkdir -p /content/FakeNewsDetector/data
!cp /content/drive/MyDrive/fakeddit/data/*.tsv /content/FakeNewsDetector/data/

In [17]:
!python scripts/make_subset.py \
  --n 12000 \
  --data_dir "/content/drive/MyDrive/fakeddit/data"

/content/FakeNewsDetector/scripts/make_subset.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=42))
  train: 12,000 rows -> /content/drive/MyDrive/fakeddit/data/subset_train.tsv  (labels: {0: 6000, 1: 6000})
/content/FakeNewsDetector/scripts/make_subset.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(

In [18]:
!python src/train.py \
  --data_dir "/content/drive/MyDrive/fakeddit/data" \
  --image_dir "/content/drive/MyDrive/fakeddit/images" \
  --checkpoint_dir "/content/drive/MyDrive/fakeddit/checkpoints" \
  --subset \
  --epochs 2 \
  --batch_size 8

Device: cuda
GPU: Tesla T4

Building dataloaders...
preprocessor_config.json: 100% 316/316 [00:00<00:00, 2.02MB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
tokenizer_config.json: 100% 905/905 [00:00<00:00, 6.21MB/s]
vocab.json: 961kB [00:00, 33.7MB/s]
merges.txt: 525kB [00:00, 122MB/s]
tokenizer.json: 2.22MB [00:00, 159MB/s]
special_tokens_map.json: 100% 389/389 [00:00<00:00, 2.84MB/s]
FakedditDataset: 11,965 samples  (/content/drive/MyDrive/fakeddit/data/subset_train.tsv)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is 

In [19]:
!python src/train.py \
  --data_dir "/content/drive/MyDrive/fakeddit/data" \
  --image_dir "/content/drive/MyDrive/fakeddit/images" \
  --checkpoint_dir "/content/drive/MyDrive/fakeddit/checkpoints" \
  --subset \
  --epochs 10 \
  --batch_size 32

Device: cuda
GPU: Tesla T4

Building dataloaders...
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
FakedditDataset: 11,965 samples  (/content/drive/MyDrive/fakeddit/data/subset_train.tsv)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  train: 11,965 samples → 373 batches of 32
FakedditDataset: 11,966 

In [20]:
path = "/content/FakeNewsDetector/src/train.py"
with open(path, "r") as f:
    content = f.read()

content = content.replace(
    "ckpt = torch.load(checkpoint_path, map_location=device)",
    "ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)"
)

with open(path, "w") as f:
    f.write(content)

print("Done — train.py patched")

Done — train.py patched


In [21]:
!python src/train.py \
  --data_dir "/content/drive/MyDrive/fakeddit/data" \
  --image_dir "/content/drive/MyDrive/fakeddit/images" \
  --checkpoint_dir "/content/drive/MyDrive/fakeddit/checkpoints" \
  --subset \
  --epochs 10 \
  --batch_size 32

Device: cuda
GPU: Tesla T4

Building dataloaders...
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
FakedditDataset: 11,965 samples  (/content/drive/MyDrive/fakeddit/data/subset_train.tsv)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  train: 11,965 samples → 373 batches of 32
FakedditDataset: 11,966 

In [22]:
path = "/content/FakeNewsDetector/src/evaluate.py"
with open(path, "r") as f:
    content = f.read()

content = content.replace(
    "ckpt = torch.load(args.checkpoint, map_location=device)",
    "ckpt = torch.load(args.checkpoint, map_location=device, weights_only=False)"
)

with open(path, "w") as f:
    f.write(content)

print("Done — evaluate.py patched")

Done — evaluate.py patched


In [24]:
path = "/content/FakeNewsDetector/src/evaluate.py"
with open(path, "r") as f:
    content = f.read()

content = content.replace(
    "model.load_state_dict(ckpt[\"model\"])",
    "state_dict = {k: v for k, v in ckpt[\"model\"].items() if not k.startswith(\"loss_fn\")}\n    model.load_state_dict(state_dict)"
)

with open(path, "w") as f:
    f.write(content)

print("Done — evaluate.py patched")

Done — evaluate.py patched


In [25]:
!python src/evaluate.py \
  --checkpoint "/content/drive/MyDrive/fakeddit/checkpoints/checkpoint_best.pt" \
  --data_dir "/content/drive/MyDrive/fakeddit/data" \
  --image_dir "/content/drive/MyDrive/fakeddit/images"

Device: cuda

Checkpoint from epoch 10
  Saved val metrics — F1: 0.8452  AUC: 0.9203

Loading weights: 100% 398/398 [00:00<00:00, 790.98it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
FakedditDataset: 883 samples  (/content/drive/MyDrive/fakeddit/data/multimodal_test_public.tsv)
/c